# Mygrate Interactive Runner
This notebook sets up the environment and exposes a single interactive cell to invoke the Mygrate orchestrator (LLM-driven) or a deterministic pipeline.
- Imports and setup are done automatically.
- Provide a `project_path` and whether to use LLM.
- Visualizations are optional and will be generated if `migration_intelligence.json` is present.

In [ ]:
import os
import sys
import json
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown

repo_root = Path.cwd()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DEBUG = True


def check_javac():
    try:
        proc = subprocess.run(['javac', '-version'], capture_output=True, text=True, check=False)
        message = (proc.stderr or proc.stdout).strip()
        return proc.returncode == 0, message or 'javac available'
    except FileNotFoundError:
        return False, 'javac not found in PATH'


def pretty(obj):
    try:
        return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

Setup complete. Use the next cell to run the pipeline interactively.


In [5]:
# Interactive runner cell
# One input only: you can enter either plain instruction text,
# or a compact command string like:
# project=freshbrew_data/cantor; llm; debug; viz; instruction=Quét dự án và sinh ma trận tương thích

def parse_run_spec(raw: str):
    text = (raw or '').strip()
    defaults = {
        'project': 'freshbrew_data/cantor',
        'use_llm': False,
        'visualize': False,
        'debug': True,
        'instruction': 'Quét dự án và sinh ma trận tương thích',
    }
    if not text:
        return defaults

    # Plain text without '=' is treated as the instruction.
    if '=' not in text:
        defaults['instruction'] = text
        return defaults

    parts = [part.strip() for part in text.split(';') if part.strip()]
    for part in parts:
        lower = part.lower()
        if lower in {'llm', 'use_llm', 'orchestrator'}:
            defaults['use_llm'] = True
            continue
        if lower in {'local', 'deterministic', 'tools', 'no-llm', 'nolllm'}:
            defaults['use_llm'] = False
            continue
        if lower in {'viz', 'visualize', 'visualise'}:
            defaults['visualize'] = True
            continue
        if lower in {'noviz', 'no-viz', 'no-visualize', 'no-visualise'}:
            defaults['visualize'] = False
            continue
        if lower in {'debug', 'dbg'}:
            defaults['debug'] = True
            continue
        if lower in {'nodebug', 'no-debug'}:
            defaults['debug'] = False
            continue
        if part.startswith('project='):
            defaults['project'] = part.split('=', 1)[1].strip() or defaults['project']
            continue
        if part.startswith('instruction='):
            defaults['instruction'] = part.split('=', 1)[1].strip() or defaults['instruction']
            continue
        if part.startswith('viz='):
            defaults['visualize'] = part.split('=', 1)[1].strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
            continue
        if part.startswith('debug='):
            defaults['debug'] = part.split('=', 1)[1].strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
            continue
        if part.startswith('llm='):
            defaults['use_llm'] = part.split('=', 1)[1].strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
            continue

    # If no instruction key was provided, treat the last free-form segment as instruction.
    free_text_segments = [part for part in parts if '=' not in part and part.lower() not in {'llm', 'use_llm', 'orchestrator', 'local', 'deterministic', 'tools', 'no-llm', 'nolllm', 'viz', 'visualize', 'visualise', 'noviz', 'no-viz', 'no-visualize', 'no-visualise', 'debug', 'dbg', 'nodebug', 'no-debug'}]
    if free_text_segments and defaults['instruction'] == 'Quét dự án và sinh ma trận tương thích':
        defaults['instruction'] = free_text_segments[-1]

    return defaults


raw_spec = input('Run spec [project=freshbrew_data/cantor; llm; debug; viz; instruction=...]: ').strip()
run_spec = parse_run_spec(raw_spec)
project = run_spec['project']
use_llm = run_spec['use_llm']
visualize = run_spec['visualize']
debug = run_spec['debug']
instruction = run_spec['instruction']

print('[+] Checking javac...')
javac_ok, javac_msg = check_javac()
print('javac:', javac_ok, javac_msg)

result = None
workflow_payload = None
workflow_ok = False

if debug:
    print('\n[DEBUG] Notebook run spec =')
    print(json.dumps(run_spec, indent=2, ensure_ascii=False))

if use_llm and HAVE_WORKFLOW:
    print('[+] Attempting LLM-driven orchestrator (Supervisor)...')
    workflow_ok, workflow_payload = run_with_workflow(project, instruction)
    if debug:
        print(f'\n[DEBUG] workflow_ok = {workflow_ok}')
        print('[DEBUG] workflow_payload =')
        print(workflow_payload)

    payload_is_useful = bool(workflow_payload) and str(workflow_payload).strip() and 'Lỗi xử lý JSON từ Supervisor.' not in str(workflow_payload)
    if workflow_ok and payload_is_useful:
        print('[+] Orchestrator completed. Full agent output follows:')
        print(workflow_payload)
        result = workflow_payload
    else:
        print('[!] Orchestrator did not produce a usable result. Falling back to deterministic pipeline.')
        rc, intel = run_local_pipeline(project)
        print(f'[+] Deterministic pipeline returned code {rc}')
        print(pretty(intel))
        result = intel
else:
    print('[+] Running deterministic pipeline (tools-only)')
    rc, intel = run_local_pipeline(project)
    print(f'[+] Deterministic pipeline returned code {rc}')
    print(pretty(intel))
    result = intel

if visualize:
    visualize_if_present('migration_intelligence.json', ask_user=True)

_mygrate_last_result = result
print('[+] Done. The variable `_mygrate_last_result` contains the returned object (or string).')

[+] Checking javac...
javac: True javac 17.0.12

[DEBUG] Notebook run spec =
{
  "project": "freshbrew_data/cantor",
  "use_llm": false,
  "visualize": false,
  "debug": true,
  "instruction": "freshbew/cantor"
}
[+] Running deterministic pipeline (tools-only)

[DEBUG] calling deterministic pipeline: d:\capstone_project\MYGRATE---Capstone-Project\scripts\run_local_pipeline.py :: run('freshbrew_data/cantor')
Scanning project: freshbrew_data\cantor
No main build file found. Exiting.

[DEBUG] migration_intelligence.json =
{
  "dependencies": [
    {
      "groupId": "junit",
      "artifactId": "junit",
      "version": "4.13.2",
      "verification_url": "https://mvnrepository.com/artifact/junit/junit/4.13.2"
    },
    {
      "groupId": "org.slf4j",
      "artifactId": "slf4j-api",
      "version": "1.7.36",
      "verification_url": "https://mvnrepository.com/artifact/org.slf4j/slf4j-api/1.7.36"
    },
    {
      "groupId": "org.apache.hadoop",
      "artifactId": "hadoop-common",
  